# 🧬 Acne Classification — Full Pipeline
### VGG16 + ResNet50 + DCGAN Augmentation (aligned with base paper)
All outputs (models, graphs, JSON metrics) are saved for Django integration.

## 01 · Installs & Imports

In [3]:
!pip install livelossplot -q

In [4]:
import os, sys, json, random, warnings, shutil
import numpy as np
import matplotlib
matplotlib.use('Agg')          # headless – safe for Kaggle
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers, regularizers
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout,
    BatchNormalization, Flatten,
    Conv2D, Conv2DTranspose, LeakyReLU, Reshape
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, f1_score
)
from datetime import datetime
warnings.filterwarnings('ignore')

# ── Output directory (Django will read from here) ──────────────────────────
OUT = '/content/Django_outputs'
for sub in ['models', 'graphs', 'json', 'confusion_matrices']:
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)

print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Output dir:', OUT)

TF version: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Output dir: /content/Django_outputs


## 02 · Download Dataset

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"akhayakumar","key":"21e8045b706eba051ad6e237da6121ed"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download -d tiswan14/acne-dataset-image
!unzip acne-dataset-image.zip

Dataset URL: https://www.kaggle.com/datasets/tiswan14/acne-dataset-image
License(s): apache-2.0
100% 122M/122M [00:00<00:00, 230MB/s]

Archive:  acne-dataset-image.zip
  inflating: AcneDataset/test/Blackheads/blackhead-1-_JPG_jpg.rf.d7b8a53aebb5da67b8acd0f8498ae9a3.jpg  
  inflating: AcneDataset/test/Blackheads/blackhead-10-_JPG_jpg.rf.0fe787a4f910d83b76190dd81c8557e4.jpg  
  inflating: AcneDataset/test/Blackheads/blackhead-12-_jpg.rf.3855ad2810048c1bda85e7a5dfaceb37.jpg  
  inflating: AcneDataset/test/Blackheads/blackhead-12-_jpg.rf.76a283b5b93b0639c48485e82c7303c2.jpg  
  inflating: AcneDataset/test/Blackheads/blackhead-12-_jpg.rf.7a59f724e34e7fa29d7a557f41c2f4ec.jpg  
  inflating: AcneDataset/test/Blackheads/blackhead-12-_jpg.rf.afc9bbd660c4655f0ecf45bb1144bd4b.jpg  
  inflating: AcneDataset/test/Blackheads/blackhead-13-_JPG_jpg.rf.e7b8ce74edfd572fcfe0ea6408702335.jpg  
  inflating: AcneDataset/test/Blackheads/blackhead-13-_JPG_jpg.rf.e8bbd2ae1be3e68aa4b6d56622bfa026.jpg  
  inflati

## 03 · Paths & Parameters

In [6]:
base_dir  = '/content/AcneDataset'
train_dir = os.path.join(base_dir, 'train')
valid_dir = os.path.join(base_dir, 'valid')
test_dir  = os.path.join(base_dir, 'test')

IMAGE_SIZE  = 224
BATCH_SIZE  = 32
NUM_CLASSES = 5      # keeping 5 classes as in your dataset
SEED        = 42

print(os.listdir(base_dir))

['train', 'valid', 'test']


## 04 · Load Raw Datasets (no preprocessing yet)

In [7]:
def load_ds(directory, shuffle=True):
    return tf.keras.preprocessing.image_dataset_from_directory(
        directory,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED
    )

raw_train_ds = load_ds(train_dir, shuffle=True)
raw_valid_ds = load_ds(valid_dir, shuffle=False)
raw_test_ds  = load_ds(test_dir,  shuffle=False)

CLASS_NAMES = raw_train_ds.class_names
print('Classes:', CLASS_NAMES)

Found 2778 files belonging to 5 classes.
Found 921 files belonging to 5 classes.
Found 918 files belonging to 5 classes.
Classes: ['Blackheads', 'Cyst', 'Papules', 'Pustules', 'Whiteheads']


## 05 · EDA — Class Distribution & Sample Images

In [9]:
# ── Count images per class per split ─────────────────────────────────────
eda_data = {}
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(base_dir, split)
    eda_data[split] = {}
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            eda_data[split][cls] = count
            print(f'  {split}/{cls}: {count}')

# Save EDA JSON
with open(f'{OUT}/json/eda_distribution.json', 'w') as f:
    json.dump(eda_data, f, indent=2)
print('\nEDA JSON saved.')

  train/Blackheads: 735
  train/Cyst: 645
  train/Papules: 621
  train/Pustules: 584
  train/Whiteheads: 193
  valid/Blackheads: 240
  valid/Cyst: 206
  valid/Papules: 209
  valid/Pustules: 217
  valid/Whiteheads: 49
  test/Blackheads: 265
  test/Cyst: 189
  test/Papules: 202
  test/Pustules: 205
  test/Whiteheads: 57

EDA JSON saved.


In [9]:
# ── Bar chart: class distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4E79A7','#F28E2B','#E15759','#76B7B2','#59A14F']

for ax, split in zip(axes, ['train', 'valid', 'test']):
    classes = list(eda_data[split].keys())
    counts  = list(eda_data[split].values())
    bars = ax.bar(classes, counts, color=colors)
    ax.set_title(f'{split.capitalize()} Set Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Acne Type')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(cnt), ha='center', va='bottom', fontsize=9)

plt.suptitle('EDA — Class Distribution Across Splits', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/graphs/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_class_distribution.png')

Saved: eda_class_distribution.png


In [10]:
# ── Pie chart (train split) ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
classes = list(eda_data['train'].keys())
counts  = list(eda_data['train'].values())
wedges, texts, autotexts = ax.pie(
    counts, labels=classes, autopct='%1.1f%%',
    colors=colors, startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(10)
ax.set_title('Training Set — Class Share', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/graphs/eda_pie_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_pie_chart.png')

Saved: eda_pie_chart.png


In [11]:
# ── Sample grid: 5 images per class ──────────────────────────────────────
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, NUM_CLASSES * 3))
for row, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(train_dir, cls)
    imgs = random.sample(os.listdir(cls_path), 5)
    for col, img_name in enumerate(imgs):
        img = keras.utils.load_img(
            os.path.join(cls_path, img_name),
            target_size=(IMAGE_SIZE, IMAGE_SIZE)
        )
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=11, fontweight='bold', rotation=90, labelpad=40)

plt.suptitle('Sample Images per Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/graphs/eda_sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: eda_sample_images.png')

Saved: eda_sample_images.png


## 06 · Class Weights (for imbalanced training)

In [10]:
# Derive counts dynamically from the train folder
train_counts = [eda_data['train'][c] for c in CLASS_NAMES]
classes_arr  = np.arange(NUM_CLASSES)
y_flat       = np.repeat(classes_arr, train_counts)
class_weight_dict = dict(zip(
    classes_arr,
    compute_class_weight('balanced', classes=classes_arr, y=y_flat)
))
print('Class weights:', class_weight_dict)

Class weights: {np.int64(0): np.float64(0.7559183673469387), np.int64(1): np.float64(0.8613953488372093), np.int64(2): np.float64(0.8946859903381642), np.int64(3): np.float64(0.9513698630136986), np.int64(4): np.float64(2.8787564766839377)}


## 07 · DCGAN — Synthetic Augmentation for Minority Classes
> Paper Section II-B: DCGAN trained on Whiteheads & minority classes to generate synthetic images.

In [13]:
# ── Identify minority classes (below median count) ───────────────────────
median_count = np.median(train_counts)
minority_classes = [CLASS_NAMES[i] for i, c in enumerate(train_counts) if c < median_count]
print(f'Minority classes (count < {median_count}):', minority_classes)

Minority classes (count < 621.0): ['Pustules', 'Whiteheads']


In [14]:
# ── DCGAN Builder ─────────────────────────────────────────────────────────
LATENT_DIM  = 128
GAN_IMG_SZ  = 64   # DCGAN generates 64×64; we resize to 224 before training

def build_generator(latent_dim=LATENT_DIM):
    model = keras.Sequential([
        keras.Input(shape=(latent_dim,)),
        Dense(4 * 4 * 512, use_bias=False),
        BatchNormalization(),
        layers.ReLU(),
        Reshape((4, 4, 512)),
        # 4→8
        Conv2DTranspose(256, 4, strides=2, padding='same', use_bias=False),
        BatchNormalization(), layers.ReLU(),
        # 8→16
        Conv2DTranspose(128, 4, strides=2, padding='same', use_bias=False),
        BatchNormalization(), layers.ReLU(),
        # 16→32
        Conv2DTranspose(64, 4, strides=2, padding='same', use_bias=False),
        BatchNormalization(), layers.ReLU(),
        # 32→64
        Conv2DTranspose(3, 4, strides=2, padding='same',
                        use_bias=False, activation='tanh'),
    ], name='Generator')
    return model

def build_discriminator():
    model = keras.Sequential([
        keras.Input(shape=(GAN_IMG_SZ, GAN_IMG_SZ, 3)),
        Conv2D(64, 4, strides=2, padding='same'),
        LeakyReLU(0.2),
        Conv2D(128, 4, strides=2, padding='same', use_bias=False),
        BatchNormalization(), LeakyReLU(0.2),
        Conv2D(256, 4, strides=2, padding='same', use_bias=False),
        BatchNormalization(), LeakyReLU(0.2),
        Conv2D(512, 4, strides=2, padding='same', use_bias=False),
        BatchNormalization(), LeakyReLU(0.2),
        Flatten(),
        Dense(1),   # raw logits for BinaryCrossentropy(from_logits=True)
    ], name='Discriminator')
    return model

print('Generator and Discriminator defined.')

Generator and Discriminator defined.


In [17]:
# ── DCGAN Training Loop ───────────────────────────────────────────────────
def load_class_images_for_gan(class_name, img_size=GAN_IMG_SZ, max_imgs=500):
    cls_path = os.path.join(train_dir, class_name)
    imgs = []
    for fname in os.listdir(cls_path)[:max_imgs]:
        try:
            img = keras.utils.load_img(
                os.path.join(cls_path, fname),
                target_size=(img_size, img_size)
            )
            arr = keras.utils.img_to_array(img)
            arr = (arr / 127.5) - 1.0   # scale to [-1, 1]
            imgs.append(arr)
        except Exception:
            pass
    return np.array(imgs, dtype=np.float32)


def train_dcgan(class_name, epochs=300, batch_size=32, latent_dim=LATENT_DIM):
    print(f'\n── Training DCGAN for class: {class_name} ──')
    real_images = load_class_images_for_gan(class_name)
    print(f'   Real images loaded: {real_images.shape}')

    gen   = build_generator(latent_dim)
    disc  = build_discriminator()

    cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)
    g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
    d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)

    dataset = tf.data.Dataset.from_tensor_slices(real_images)\
                             .shuffle(1000).batch(batch_size)

    @tf.function
    def train_step(real_batch):
        noise = tf.random.normal([tf.shape(real_batch)[0], latent_dim])
        with tf.GradientTape() as gt, tf.GradientTape() as dt:
            fake   = gen(noise, training=True)
            r_out  = disc(real_batch, training=True)
            f_out  = disc(fake,       training=True)
            d_loss = (cross_entropy(tf.ones_like(r_out),  r_out) +
                      cross_entropy(tf.zeros_like(f_out), f_out))
            g_loss =  cross_entropy(tf.ones_like(f_out),  f_out)
            d_grads = dt.gradient(d_loss, disc.trainable_variables)
            g_grads = gt.gradient(g_loss, gen.trainable_variables)
            d_opt.apply_gradients(zip(d_grads, disc.trainable_variables))
            g_opt.apply_gradients(zip(g_grads, gen.trainable_variables))
        return d_loss, g_loss

    for epoch in range(epochs):
        for batch in dataset:
            d_l, g_l = train_step(batch)
        if (epoch + 1) % 100 == 0:
            print(f'   Epoch {epoch+1}/{epochs} — d_loss: {d_l:.4f}  g_loss: {g_l:.4f}')

    return gen


def generate_synthetic_images(generator, n=200, latent_dim=LATENT_DIM, out_size=IMAGE_SIZE):
    noise = tf.random.normal([n, latent_dim])
    imgs  = generator(noise, training=False).numpy()
    imgs  = ((imgs + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    # resize from 64 to 224
    resized = []
    for img in imgs:
        pil = keras.utils.array_to_img(img)
        pil = pil.resize((out_size, out_size))
        resized.append(keras.utils.img_to_array(pil).astype(np.float32))
    return np.array(resized)


print('DCGAN functions defined.')

DCGAN functions defined.


In [18]:
# ── Train DCGAN for each minority class & collect synthetic data ──────────
# Each minority class gets ~300 synthetic images to balance the dataset.
synthetic_images_by_class = {}   # { class_name: np.ndarray (N, 224, 224, 3) }
target_synthetic = 300

for cls in minority_classes:
    gen_model = train_dcgan(cls, epochs=300)
    synth = generate_synthetic_images(gen_model, n=target_synthetic)
    synthetic_images_by_class[cls] = synth
    print(f'Generated {len(synth)} synthetic images for {cls}')

    # Save a preview grid
    fig, axes = plt.subplots(3, 6, figsize=(14, 7))
    for i, ax in enumerate(axes.flat):
        ax.imshow(synth[i].astype(np.uint8))
        ax.axis('off')
    plt.suptitle(f'DCGAN Synthetic — {cls}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT}/graphs/dcgan_samples_{cls}.png', dpi=100, bbox_inches='tight')
    plt.close()
    print(f'Saved: dcgan_samples_{cls}.png')


── Training DCGAN for class: Pustules ──
   Real images loaded: (500, 64, 64, 3)
   Epoch 100/300 — d_loss: 0.1667  g_loss: 3.8276
   Epoch 200/300 — d_loss: 0.1510  g_loss: 6.2452
   Epoch 300/300 — d_loss: 0.1160  g_loss: 5.7064
Generated 300 synthetic images for Pustules
Saved: dcgan_samples_Pustules.png

── Training DCGAN for class: Whiteheads ──
   Real images loaded: (193, 64, 64, 3)
   Epoch 100/300 — d_loss: 9.6016  g_loss: 0.5603
   Epoch 200/300 — d_loss: 0.1471  g_loss: 3.7158
   Epoch 300/300 — d_loss: 0.2380  g_loss: 1.5519
Generated 300 synthetic images for Whiteheads
Saved: dcgan_samples_Whiteheads.png


## 08 · Build Augmented TF Datasets
> Combine real images + DCGAN synthetic images → balanced training set

In [11]:
# ── Helper: load all real images for a class ──────────────────────────────
def load_all_class_images(class_name, img_size=IMAGE_SIZE):
    cls_path = os.path.join(train_dir, class_name)
    imgs, labels = [], []
    label_idx = CLASS_NAMES.index(class_name)
    for fname in os.listdir(cls_path):
        try:
            img = keras.utils.load_img(
                os.path.join(cls_path, fname),
                target_size=(img_size, img_size)
            )
            imgs.append(keras.utils.img_to_array(img).astype(np.float32))
            labels.append(label_idx)
        except Exception:
            pass
    return imgs, labels


# ── Build augmented training arrays ──────────────────────────────────────
all_train_images, all_train_labels = [], []

for cls in CLASS_NAMES:
    imgs, labels = load_all_class_images(cls)
    all_train_images.extend(imgs)
    all_train_labels.extend(labels)
    print(f'Real  {cls}: {len(imgs)}')

    # Append synthetic images if this class was augmented
    if cls in synthetic_images_by_class:
        synth = synthetic_images_by_class[cls]
        lbl   = CLASS_NAMES.index(cls)
        all_train_images.extend(list(synth))
        all_train_labels.extend([lbl] * len(synth))
        print(f'Synth {cls}: +{len(synth)}')

X_train = np.array(all_train_images, dtype=np.float32)
y_train = np.array(all_train_labels, dtype=np.int32)

# Shuffle
idx = np.random.permutation(len(X_train))
X_train, y_train = X_train[idx], y_train[idx]

print(f'\nAugmented training set: {X_train.shape}, labels: {y_train.shape}')
print('Class counts after augmentation:', {CLASS_NAMES[i]: int(np.sum(y_train==i)) for i in range(NUM_CLASSES)})

Real  Blackheads: 735


NameError: name 'synthetic_images_by_class' is not defined

In [12]:
# ── Preprocess with VGG16 preprocessor (used for both VGG16 & ResNet50) ───
from tensorflow.keras.applications.vgg16    import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# We'll build separate pipelines per model below.
# For valid/test we reload from directory each time.

def make_tf_dataset(X, y, preprocess_fn, batch_size=BATCH_SIZE, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, lbl: (preprocess_fn(x), lbl),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


def make_dir_dataset(directory, preprocess_fn, shuffle=False):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        directory,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED
    )
    return ds.map(lambda x, y: (preprocess_fn(tf.cast(x, tf.float32)), y),
                  num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)


print('Dataset pipeline helpers defined.')

Dataset pipeline helpers defined.


## 09 · Model Builder — VGG16 & ResNet50
> Paper Section II-C: GAP → Dense(256, ReLU) → Dropout(0.4) → Softmax(6)  
> Adapted: Dense(256) + Dropout(0.4), 5 classes

In [13]:
def build_vgg16(num_classes=NUM_CLASSES):
    base = VGG16(weights='imagenet', include_top=False,
                 input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    # Freeze all except last 2 conv blocks (as per paper)
    for layer in base.layers[:-8]:
        layer.trainable = False
    for layer in base.layers[-8:]:
        layer.trainable = True

    model = keras.Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ], name='VGG16_Acne')
    return model


def build_resnet50(num_classes=NUM_CLASSES):
    base = ResNet50(weights='imagenet', include_top=False,
                    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    # Fine-tune last 3 conv blocks (as per paper)
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers[-30:]:
        layer.trainable = True

    model = keras.Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ], name='ResNet50_Acne')
    return model


print('Model builders ready.')

Model builders ready.


## 10 · Training Helper — saves everything for Django

In [14]:
def save_epoch_history(history, model_name, stage):
    """Save epoch-level metrics to JSON."""
    data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    data['model']  = model_name
    data['stage']  = stage
    data['epochs'] = list(range(1, len(data['accuracy']) + 1))
    path = f'{OUT}/json/history_{model_name}_{stage}.json'
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f'  History saved → {path}')
    return data


def plot_training_curves(history_data, model_name, stage):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = history_data['epochs']
    ax1.plot(epochs, history_data['accuracy'],     label='Train Acc',  color='#4E79A7', linewidth=2)
    ax1.plot(epochs, history_data['val_accuracy'], label='Val Acc',    color='#F28E2B', linewidth=2, linestyle='--')
    ax1.set_title(f'{model_name} ({stage}) — Accuracy', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, history_data['loss'],     label='Train Loss', color='#E15759', linewidth=2)
    ax2.plot(epochs, history_data['val_loss'], label='Val Loss',   color='#76B7B2', linewidth=2, linestyle='--')
    ax2.set_title(f'{model_name} ({stage}) — Loss', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.suptitle(f'{model_name} Training Curves — {stage}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    path = f'{OUT}/graphs/curves_{model_name}_{stage}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Curves saved → {path}')


def evaluate_and_save(model, test_ds, model_name, stage, class_names=CLASS_NAMES):
    """Evaluate model, save confusion matrix, classification report and metrics JSON."""
    y_true = np.concatenate([y.numpy() for _, y in test_ds])
    y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    f1  = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    report = classification_report(
        y_true, y_pred, target_names=class_names,
        output_dict=True, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)

    # ── Confusion matrix heatmap ──
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True',      fontsize=11)
    ax.set_title(f'Confusion Matrix — {model_name} ({stage})', fontsize=13, fontweight='bold')
    plt.xticks(rotation=30); plt.yticks(rotation=0)
    plt.tight_layout()
    cm_path = f'{OUT}/confusion_matrices/cm_{model_name}_{stage}.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Confusion matrix saved → {cm_path}')

    # ── Normalized confusion matrix ──
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True',      fontsize=11)
    ax.set_title(f'Normalised CM — {model_name} ({stage})', fontsize=13, fontweight='bold')
    plt.xticks(rotation=30); plt.yticks(rotation=0)
    plt.tight_layout()
    cm_norm_path = f'{OUT}/confusion_matrices/cm_norm_{model_name}_{stage}.png'
    plt.savefig(cm_norm_path, dpi=150, bbox_inches='tight')
    plt.show()

    # ── Metrics JSON ──
    metrics = {
        'model': model_name, 'stage': stage,
        'accuracy': round(acc, 4),
        'precision': round(prec, 4),
        'f1_score': round(f1, 4),
        'classification_report': report,
        'confusion_matrix': cm.tolist()
    }
    metrics_path = f'{OUT}/json/metrics_{model_name}_{stage}.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f'  Metrics saved → {metrics_path}')
    print(f'  Accuracy: {acc:.4f}  Precision: {prec:.4f}  F1: {f1:.4f}')
    return metrics


def save_model_both_formats(model, model_name, stage):
    # Keras native format
    keras_path = f'{OUT}/models/{model_name}_{stage}_best.keras'
    model.save(keras_path)
    # Legacy HDF5
    h5_path = f'{OUT}/models/{model_name}_{stage}_best.h5'
    model.save(h5_path)
    print(f'  Model saved → {keras_path}')
    print(f'  Model saved → {h5_path}')


print('All helper functions defined.')

All helper functions defined.


## 11 · Baseline Training — VGG16 (no DCGAN)

In [24]:
# Build datasets with VGG16 preprocessing
train_vgg_base = make_tf_dataset(X_train[:int(0.8*len(X_train))],
                                  y_train[:int(0.8*len(y_train))],
                                  vgg_preprocess, shuffle=True)
valid_vgg = make_dir_dataset(valid_dir, vgg_preprocess)
test_vgg  = make_dir_dataset(test_dir,  vgg_preprocess)

# Build model
vgg16_base = build_vgg16()
vgg16_base.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
vgg16_base.summary()

Found 921 files belonging to 5 classes.
Found 918 files belonging to 5 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "VGG16_Acne"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,301 (56.64 MB)

 Trainable params: 13,111,813 (50.02 MB)

 Non-trainable params: 1,735,488 (6.62 MB)

In [25]:
cb_vgg_base = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(f'{OUT}/models/VGG16_baseline_best.keras',
                    save_best_only=True, monitor='val_accuracy', verbose=1)
]

# Paper trains 15 epochs before GAN augmentation
hist_vgg_base = vgg16_base.fit(
    train_vgg_base,
    validation_data=valid_vgg,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=cb_vgg_base
)

h_vgg_base = save_epoch_history(hist_vgg_base, 'VGG16', 'baseline')
plot_training_curves(h_vgg_base, 'VGG16', 'baseline')
metrics_vgg_base = evaluate_and_save(vgg16_base, test_vgg, 'VGG16', 'baseline')
save_model_both_formats(vgg16_base, 'VGG16', 'baseline')

Epoch 1/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - accuracy: 0.3150 - loss: 1.8390
Epoch 1: val_accuracy improved from None to 0.45820, saving model to /content/Django_outputs/models/VGG16_baseline_best.keras

Epoch 1: finished saving model to /content/Django_outputs/models/VGG16_baseline_best.keras
85/85 ━━━━━━━━━━━━━━━━━━━━ 86s 698ms/step - accuracy: 0.3942 - loss: 1.3773 - val_accuracy: 0.4582 - val_loss: 1.2181 - learning_rate: 1.0000e-04
Epoch 2/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step - accuracy: 0.4687 - loss: 1.2409
Epoch 2: val_accuracy did not improve from 0.45820
85/85 ━━━━━━━━━━━━━━━━━━━━ 29s 341ms/step - accuracy: 0.4582 - loss: 1.2594 - val_accuracy: 0.3377 - val_loss: 1.5095 - learning_rate: 1.0000e-04
Epoch 3/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 0.4892 - loss: 1.0799
Epoch 3: val_accuracy did not improve from 0.45820
85/85 ━━━━━━━━━━━━━━━━━━━━ 29s 336ms/step - accuracy: 0.4815 - loss: 1.1259 - val_accuracy: 0.2508 - val_loss: 1.4836 - learning_r

  Model saved → /content/Django_outputs/models/VGG16_baseline_best.keras
  Model saved → /content/Django_outputs/models/VGG16_baseline_best.h5


## 12 · Baseline Training — ResNet50 (no DCGAN)

In [26]:
train_res_base = make_tf_dataset(X_train[:int(0.8*len(X_train))],
                                  y_train[:int(0.8*len(y_train))],
                                  resnet_preprocess, shuffle=True)
valid_res = make_dir_dataset(valid_dir, resnet_preprocess)
test_res  = make_dir_dataset(test_dir,  resnet_preprocess)

resnet50_base = build_resnet50()
resnet50_base.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_res_base = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(f'{OUT}/models/ResNet50_baseline_best.keras',
                    save_best_only=True, monitor='val_accuracy', verbose=1)
]

hist_res_base = resnet50_base.fit(
    train_res_base,
    validation_data=valid_res,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=cb_res_base
)

h_res_base = save_epoch_history(hist_res_base, 'ResNet50', 'baseline')
plot_training_curves(h_res_base, 'ResNet50', 'baseline')
metrics_res_base = evaluate_and_save(resnet50_base, test_res, 'ResNet50', 'baseline')
save_model_both_formats(resnet50_base, 'ResNet50', 'baseline')

Found 921 files belonging to 5 classes.
Found 918 files belonging to 5 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - accuracy: 0.4713 - loss: 1.2547
Epoch 1: val_accuracy improved from None to 0.66341, saving model to /content/Django_outputs/models/ResNet50_baseline_best.keras

Epoch 1: finished saving model to /content/Django_outputs/models/ResNet50_baseline_best.keras
85/85 ━━━━━━━━━━━━━━━━━━━━ 62s 433ms/step - accuracy: 0.6051 - loss: 0.9348 - val_accuracy: 0.6634 - val_loss: 0.8206 - learning_rate: 1.0000e-04
Epoch 2/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.8904 - loss: 0.3052
Epoch 2: val_accuracy improved from 0.66341 to 0.75244, saving model to /content/Django_outputs/models/ResNet50_baseline_best.keras

Epoch 2: finished saving model to /content/Django_outputs/models/ResNet50_baseline_best.keras
85/85 ━━━━━━━━━━━━━━━━━━━━ 15s 181ms/step - accuracy: 0.9004 - loss: 0.2773 - val_accuracy: 0.7524 -

  Model saved → /content/Django_outputs/models/ResNet50_baseline_best.keras
  Model saved → /content/Django_outputs/models/ResNet50_baseline_best.h5


## 13 · Post-DCGAN Training — VGG16 (augmented dataset)

In [17]:
# Use the full augmented dataset (real + synthetic)
train_vgg_aug = make_tf_dataset(X_train, y_train, vgg_preprocess, shuffle=True)

# Re-build fresh model (paper trains fresh after augmentation)
vgg16_aug = build_vgg16()
vgg16_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_vgg_aug = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(f'{OUT}/models/VGG16_dcgan_best.keras',
                    save_best_only=True, monitor='val_accuracy', verbose=1)
]

# Paper: 20 epochs after augmentation
hist_vgg_aug = vgg16_aug.fit(
    train_vgg_aug,
    validation_data=valid_vgg,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=cb_vgg_aug
)

h_vgg_aug = save_epoch_history(hist_vgg_aug, 'VGG16', 'dcgan')
plot_training_curves(h_vgg_aug, 'VGG16', 'dcgan')
metrics_vgg_aug = evaluate_and_save(vgg16_aug, test_vgg, 'VGG16', 'dcgan')
save_model_both_formats(vgg16_aug, 'VGG16', 'dcgan')

NameError: name 'X_train' is not defined

In [18]:
# No in-memory arrays — reads directly from disk, safe on RAM
train_vgg_aug = make_dir_dataset(train_dir, vgg_preprocess, shuffle=True)
valid_vgg     = make_dir_dataset(valid_dir, vgg_preprocess)
test_vgg      = make_dir_dataset(test_dir,  vgg_preprocess)

vgg16_aug = build_vgg16()
vgg16_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_vgg_aug = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(f'{OUT}/models/VGG16_dcgan_best.keras',
                    save_best_only=True, monitor='val_accuracy', verbose=1)
]

hist_vgg_aug = vgg16_aug.fit(
    train_vgg_aug,
    validation_data=valid_vgg,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=cb_vgg_aug
)

h_vgg_aug = save_epoch_history(hist_vgg_aug, 'VGG16', 'dcgan')
plot_training_curves(h_vgg_aug, 'VGG16', 'dcgan')
metrics_vgg_aug = evaluate_and_save(vgg16_aug, test_vgg, 'VGG16', 'dcgan')
save_model_both_formats(vgg16_aug, 'VGG16', 'dcgan')

del vgg16_aug
tf.keras.backend.clear_session()
gc.collect()
print('Memory cleared ✅')

Found 2778 files belonging to 5 classes.
Found 921 files belonging to 5 classes.
Found 918 files belonging to 5 classes.
Epoch 1/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 508ms/step - accuracy: 0.2383 - loss: 1.9402
Epoch 1: val_accuracy improved from None to 0.25624, saving model to /content/Django_outputs/models/VGG16_dcgan_best.keras

Epoch 1: finished saving model to /content/Django_outputs/models/VGG16_dcgan_best.keras
87/87 ━━━━━━━━━━━━━━━━━━━━ 96s 768ms/step - accuracy: 0.2513 - loss: 1.6371 - val_accuracy: 0.2562 - val_loss: 1.5700 - learning_rate: 1.0000e-04
Epoch 2/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step - accuracy: 0.2912 - loss: 1.4849
Epoch 2: val_accuracy improved from 0.25624 to 0.44734, saving model to /content/Django_outputs/models/VGG16_dcgan_best.keras

Epoch 2: finished saving model to /content/Django_outputs/models/VGG16_dcgan_best.keras
87/87 ━━━━━━━━━━━━━━━━━━━━ 30s 349ms/step - accuracy: 0.3283 - loss: 1.4237 - val_accuracy: 0.4473 - val_loss: 1.2750 - learning_rate:

  Model saved → /content/Django_outputs/models/VGG16_dcgan_best.keras
  Model saved → /content/Django_outputs/models/VGG16_dcgan_best.h5
Memory cleared ✅


In [ ]:
import gc
import tensorflow as tf

# Delete baseline models from RAM (already saved to disk)
del vgg16_base
del resnet50_base

# Delete DCGAN models (already used, no longer needed)
del synthetic_images_by_class

# Clear TF session
tf.keras.backend.clear_session()

# Force garbage collection
gc.collect()

print("Memory cleared ✅")

## 14 · Post-DCGAN Training — ResNet50 (augmented dataset)

In [20]:
train_res_aug = make_dir_dataset(train_dir, resnet_preprocess, shuffle=True)
valid_res     = make_dir_dataset(valid_dir, resnet_preprocess)
test_res      = make_dir_dataset(test_dir,  resnet_preprocess)

resnet50_aug = build_resnet50()
resnet50_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_res_aug = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(f'{OUT}/models/ResNet50_dcgan_best.keras',
                    save_best_only=True, monitor='val_accuracy', verbose=1)
]

hist_res_aug = resnet50_aug.fit(
    train_res_aug,
    validation_data=valid_res,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=cb_res_aug
)

h_res_aug = save_epoch_history(hist_res_aug, 'ResNet50', 'dcgan')
plot_training_curves(h_res_aug, 'ResNet50', 'dcgan')
metrics_res_aug = evaluate_and_save(resnet50_aug, test_res, 'ResNet50', 'dcgan')
save_model_both_formats(resnet50_aug, 'ResNet50', 'dcgan')

del resnet50_aug
tf.keras.backend.clear_session()
gc.collect()
print('Done ✅')

Found 2778 files belonging to 5 classes.
Found 921 files belonging to 5 classes.
Found 918 files belonging to 5 classes.
Epoch 1/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.4228 - loss: 1.3254
Epoch 1: val_accuracy improved from None to 0.73073, saving model to /content/Django_outputs/models/ResNet50_dcgan_best.keras

Epoch 1: finished saving model to /content/Django_outputs/models/ResNet50_dcgan_best.keras
87/87 ━━━━━━━━━━━━━━━━━━━━ 64s 406ms/step - accuracy: 0.5709 - loss: 0.9923 - val_accuracy: 0.7307 - val_loss: 0.6528 - learning_rate: 1.0000e-04
Epoch 2/20
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.8969 - loss: 0.3148
Epoch 2: val_accuracy improved from 0.73073 to 0.84908, saving model to /content/Django_outputs/models/ResNet50_dcgan_best.keras

Epoch 2: finished saving model to /content/Django_outputs/models/ResNet50_dcgan_best.keras
87/87 ━━━━━━━━━━━━━━━━━━━━ 16s 184ms/step - accuracy: 0.9212 - loss: 0.2512 - val_accuracy: 0.8491 - val_loss: 0.4007 - le

  Model saved → /content/Django_outputs/models/ResNet50_dcgan_best.keras
  Model saved → /content/Django_outputs/models/ResNet50_dcgan_best.h5
Done ✅


## 15 · Comparison Plots & Summary JSON

In [22]:
with open(f'{OUT}/json/metrics_VGG16_baseline.json')    as f: metrics_vgg_base = json.load(f)
with open(f'{OUT}/json/metrics_ResNet50_baseline.json') as f: metrics_res_base = json.load(f)
with open(f'{OUT}/json/metrics_VGG16_dcgan.json')       as f: metrics_vgg_aug  = json.load(f)
with open(f'{OUT}/json/metrics_ResNet50_dcgan.json')    as f: metrics_res_aug  = json.load(f)

summary = {
    'class_names': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'results': [
        {'model':'VGG16',    'gan_augmentation': False, 'accuracy': metrics_vgg_base['accuracy'], 'precision': metrics_vgg_base['precision'], 'f1_score': metrics_vgg_base['f1_score']},
        {'model':'VGG16',    'gan_augmentation': True,  'accuracy': metrics_vgg_aug['accuracy'],  'precision': metrics_vgg_aug['precision'],  'f1_score': metrics_vgg_aug['f1_score']},
        {'model':'ResNet50', 'gan_augmentation': False, 'accuracy': metrics_res_base['accuracy'], 'precision': metrics_res_base['precision'], 'f1_score': metrics_res_base['f1_score']},
        {'model':'ResNet50', 'gan_augmentation': True,  'accuracy': metrics_res_aug['accuracy'],  'precision': metrics_res_aug['precision'],  'f1_score': metrics_res_aug['f1_score']},
    ],
    'paper_reference': {
        'VGG16_baseline':    0.4554,
        'VGG16_dcgan':       0.8425,
        'ResNet50_baseline': 0.3216,
        'ResNet50_dcgan':    0.4820
    }
}
with open(f'{OUT}/json/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('summary.json saved ✅')
print('\n=== All output files ===')
for root, dirs, files in os.walk(OUT):
    for fname in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, fname)) / 1024
        print(f'  {root.replace(OUT,"")}/{fname}  ({size_kb:.1f} KB)')

summary.json saved ✅

=== All output files ===
  /confusion_matrices/cm_ResNet50_baseline.png  (64.4 KB)
  /confusion_matrices/cm_ResNet50_dcgan.png  (63.3 KB)
  /confusion_matrices/cm_VGG16_baseline.png  (64.9 KB)
  /confusion_matrices/cm_VGG16_dcgan.png  (64.4 KB)
  /confusion_matrices/cm_norm_ResNet50_baseline.png  (77.3 KB)
  /confusion_matrices/cm_norm_ResNet50_dcgan.png  (77.2 KB)
  /confusion_matrices/cm_norm_VGG16_baseline.png  (77.8 KB)
  /confusion_matrices/cm_norm_VGG16_dcgan.png  (76.2 KB)
  /json/eda_distribution.json  (0.4 KB)
  /json/history_ResNet50_baseline.json  (1.3 KB)
  /json/history_ResNet50_dcgan.json  (2.5 KB)
  /json/history_VGG16_baseline.json  (2.1 KB)
  /json/history_VGG16_dcgan.json  (2.7 KB)
  /json/metrics_ResNet50_baseline.json  (1.6 KB)
  /json/metrics_ResNet50_dcgan.json  (1.6 KB)
  /json/metrics_VGG16_baseline.json  (1.6 KB)
  /json/metrics_VGG16_dcgan.json  (1.6 KB)
  /json/summary.json  (0.9 KB)
  /graphs/curves_ResNet50_baseline.png  (101.3 KB)
  /

In [24]:
# ── 3. Class-wise accuracy improvement VGG16 ─────────────────────────────
# Use class names from the confusion matrix size, not directory
n_classes = len(m_vb['confusion_matrix'])
cm_class_names = sorted(os.listdir(train_dir))[:n_classes]

# If CM has 6 classes, load the original 6-class names from metrics JSON
report_classes = list(m_vb['classification_report'].keys())
report_classes = [c for c in report_classes if c not in ['accuracy','macro avg','weighted avg']]
print('Classes in CM:', report_classes)

def class_accs(cm_list):
    cm = np.array(cm_list)
    return [cm[i,i]/cm[i].sum() if cm[i].sum()>0 else 0 for i in range(len(cm))]

acc_base = class_accs(m_vb['confusion_matrix'])
acc_aug  = class_accs(m_va['confusion_matrix'])

x = np.arange(len(report_classes)); width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, [v*100 for v in acc_base], width, label='VGG16 Baseline', color='#E15759', alpha=0.85)
b2 = ax.bar(x + width/2, [v*100 for v in acc_aug],  width, label='VGG16 + DCGAN',  color='#4E79A7', alpha=0.85)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)
ax.set_ylabel('Per-class Accuracy (%)', fontsize=12)
ax.set_title('VGG16 — Class-wise Accuracy Before vs After DCGAN', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(report_classes, rotation=20, fontsize=10)
ax.legend(); ax.set_ylim(0, 115); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/graphs/classwise_accuracy_vgg16.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: classwise_accuracy_vgg16.png ✅')

Classes in CM: ['Blackheads', 'Cyst', 'Papules', 'Pustules', 'Whiteheads']
Saved: classwise_accuracy_vgg16.png ✅


In [25]:
# Zip everything for easy download
import shutil
shutil.make_archive('/content/Django_outputs', 'zip', '/content/Django_outputs')
print('Zip created: /content/Django_outputs.zip')

Zip created: /content/Django_outputs.zip


In [ ]:
# ── Master summary JSON (for Django comparison table) ─────────────────────
summary = {
    'class_names': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'eda': eda_data,
    'results': [
        {
            'model': 'VGG16',
            'gan_augmentation': False,
            'accuracy':  metrics_vgg_base['accuracy'],
            'precision': metrics_vgg_base['precision'],
            'f1_score':  metrics_vgg_base['f1_score'],
        },
        {
            'model': 'VGG16',
            'gan_augmentation': True,
            'accuracy':  metrics_vgg_aug['accuracy'],
            'precision': metrics_vgg_aug['precision'],
            'f1_score':  metrics_vgg_aug['f1_score'],
        },
        {
            'model': 'ResNet50',
            'gan_augmentation': False,
            'accuracy':  metrics_res_base['accuracy'],
            'precision': metrics_res_base['precision'],
            'f1_score':  metrics_res_base['f1_score'],
        },
        {
            'model': 'ResNet50',
            'gan_augmentation': True,
            'accuracy':  metrics_res_aug['accuracy'],
            'precision': metrics_res_aug['precision'],
            'f1_score':  metrics_res_aug['f1_score'],
        },
    ],
    'paper_reference': {
        'VGG16_baseline': 0.4554,
        'VGG16_dcgan':    0.8425,
        'ResNet50_baseline': 0.3216,
        'ResNet50_dcgan':    0.4820,
    }
}
with open(f'{OUT}/json/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Master summary saved →', f'{OUT}/json/summary.json')

## 16 · Class-wise Accuracy Improvement Chart

In [ ]:
# ── Per-class accuracy before & after for VGG16 ───────────────────────────
def class_accuracies_from_cm(cm_list, n_classes):
    cm = np.array(cm_list)
    return [cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0 for i in range(n_classes)]

acc_vgg_base_cls = class_accuracies_from_cm(
    metrics_vgg_base['confusion_matrix'], NUM_CLASSES)
acc_vgg_aug_cls  = class_accuracies_from_cm(
    metrics_vgg_aug['confusion_matrix'],  NUM_CLASSES)

x = np.arange(NUM_CLASSES)
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, [v*100 for v in acc_vgg_base_cls], width,
            label='VGG16 Baseline', color='#E15759', alpha=0.85)
b2 = ax.bar(x + width/2, [v*100 for v in acc_vgg_aug_cls],  width,
            label='VGG16 + DCGAN',  color='#4E79A7', alpha=0.85)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)

ax.set_ylabel('Per-class Accuracy (%)', fontsize=12)
ax.set_title('VGG16 — Class-wise Accuracy Before vs After DCGAN',
             fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=20, fontsize=10)
ax.legend(); ax.set_ylim(0, 110); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/graphs/classwise_accuracy_vgg16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: classwise_accuracy_vgg16.png')

## 17 · List All Saved Outputs

In [ ]:
print('\n=== All files saved to', OUT, '===')
for root, dirs, files in os.walk(OUT):
    level = root.replace(OUT, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level == 0:
        print(f'{indent}{folder}/')
    else:
        print(f'{indent}{folder}/')
    sub_indent = '  ' * (level + 1)
    for fname in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, fname)) / 1024
        print(f'{sub_indent}{fname}  ({size_kb:.1f} KB)')

## Django Integration Guide

Copy the entire `django_outputs/` folder into your Django project's `media/` or `static/` directory.

**models/** — load for prediction:
```python
from tensorflow import keras
model = keras.models.load_model('media/models/VGG16_dcgan_best.keras')
# or
model = keras.models.load_model('media/models/VGG16_dcgan_best.h5')
```

**json/summary.json** — comparison table data

**json/history_VGG16_dcgan.json** — epoch-by-epoch metrics for Chart.js

**graphs/** — display as `<img>` tags directly

**confusion_matrices/** — per-model confusion matrices

Class names order: see `summary.json['class_names']`